<a href="https://colab.research.google.com/github/murilopdr/murilo-76-tp546/blob/main/trabalho_pratico/FreezerGuard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreezerGuard — tomada inteligente para proteção de freezers

**Problema.** Uma queda de energia, uma porta mal fechada ou um compressor parado podem estragar todo o conteúdo de um freezer, e o dono só descobre quando já é tarde. Para pequenos comércios (açougues, padarias, farmácias) o prejuízo é direto.

**Solução.** Uma tomada inteligente instalada entre a parede e o freezer. Ela mede **tensão** e **potência**, lê uma **sonda de temperatura** interna, um **sensor de porta** e um **sensor de temperatura ambiente**, envia tudo por **MQTT** e publica **alarmes** quando algo sai do normal. Pelo relé, o usuário também pode **ligar e desligar** o freezer remotamente.

**Neste notebook**, o freezer é simulado em Python, a tomada se comunica com a biblioteca `paho-mqtt` e a visualização e os comandos são feitos pelo **MQTTX**.

### Configuração

##### Instalação

In [ ]:
!pip install -q "paho-mqtt>=2.0"

##### Comunicação MQTT

In [ ]:
BROKER = "broker.emqx.io"              # broker público e gratuito
PORTA = 1883
TOPICO = "inatel/murilo/freezerguard"  # prefixo único: o broker é público
INTERVALO_S = 2                        # segundos entre leituras (cada leitura = 1 minuto simulado)

##### Simulação do freezer

In [ ]:
TEMP_INICIAL = -18.0                   # temperatura interna inicial (°C)
TERMOSTATO_LIGA = -17.0                # compressor liga acima disso (°C)
TERMOSTATO_DESLIGA = -19.0             # compressor desliga abaixo disso (°C)
RESFRIAMENTO = 0.5                     # frio produzido pelo compressor (°C por minuto)
K_PORTA_FECHADA = 0.0035               # troca de calor com o ambiente, porta fechada (por minuto)
K_PORTA_ABERTA = 0.025                 # troca de calor com o ambiente, porta aberta (por minuto)

# Temperatura ambiente: sorteada uma vez (gaussiana) e constante durante a simulação
import random
TEMP_AMB_MEDIA = 25.0                  # média (°C)
TEMP_AMB_DESVIO = 2.0                  # desvio-padrão (°C)
TEMP_AMBIENTE = random.gauss(TEMP_AMB_MEDIA, TEMP_AMB_DESVIO)
print(f"Temperatura ambiente sorteada: {TEMP_AMBIENTE:.1f} °C")

Temperatura ambiente sorteada: 24.0 °C


##### Alarmes

In [ ]:
LIMITE_TEMP = -12.0                    # temperatura alta: acima deste valor (°C)
TENSAO_MINIMA = 80.0                   # falta de energia: abaixo deste valor (V)
PORTA_TEMPO_MAX_MIN = 3                # porta aberta: aviso a partir deste tempo (min)

# Gravidade de cada alarme: vai no payload MQTT e define a cor na saída
SEVERIDADE = {
    "FALTA_DE_ENERGIA": "critico",
    "TEMPERATURA_ALTA": "critico",
    "PORTA_ABERTA": "aviso",
}

### Simulação do freezer

- Cada chamada de `avancar_freezer()` simula um avanço de 1 minuto no tempo simulado. Cada minuto simulado corresponde a 2 segundos reais.
- **Termostato:** com energia, o compressor liga acima de −17 °C e desliga abaixo de −19 °C.
- **Troca de calor:** o calor que entra é proporcional à diferença entre o ambiente e o interior (lei de resfriamento de Newton), e é maior com a porta aberta. Sem energia, a temperatura sobe até se aproximar da ambiente.
- O dicionário `estado` é alterado a cada minuto e pelos comandos recebidos via MQTT.

In [ ]:
# Estado da simulação: alterado a cada minuto e pelos comandos MQTT
estado = {
    "minuto": 0,
    "temperatura": TEMP_INICIAL,  # temperatura interna real (°C)
    "compressor": False,
    "energia": True,              # há energia na rede?
    "porta_aberta": False,
    "rele": True,                 # relé da tomada (comandado remotamente)
}

def avancar_freezer():
    """Avança a simulação em 1 minuto."""
    estado["minuto"] += 1
    alimentado = estado["energia"] and estado["rele"]

    # Termostato do freezer (só funciona com energia)
    if not alimentado:
        estado["compressor"] = False
    elif estado["temperatura"] > TERMOSTATO_LIGA:
        estado["compressor"] = True
    elif estado["temperatura"] < TERMOSTATO_DESLIGA:
        estado["compressor"] = False

    # Troca de calor com o ambiente (lei de resfriamento de Newton)
    k = K_PORTA_ABERTA if estado["porta_aberta"] else K_PORTA_FECHADA
    estado["temperatura"] += k * (TEMP_AMBIENTE - estado["temperatura"])

    # Resfriamento pelo compressor
    if estado["compressor"]:
        estado["temperatura"] -= RESFRIAMENTO

### Lógica da tomada inteligente

Parte que roda no produto. A tomada só usa o que seus sensores medem; o estado do compressor, por exemplo, é inferido pela potência.

| Alarme | Condição | Gravidade |
|---|---|---|
| `FALTA_DE_ENERGIA` | tensão abaixo de 80 V | crítico |
| `TEMPERATURA_ALTA` | temperatura interna acima de −12 °C | crítico |
| `PORTA_ABERTA` | porta aberta por 3 min seguidos ou mais | aviso |

Um alarme só gera mensagem quando muda de estado (ativo ou encerrado).

In [ ]:
def ler_sensores():
    """Retorna as medições da tomada (com ruído de medição)."""
    alimentado = estado["energia"] and estado["rele"]
    if estado["compressor"]:
        potencia = random.gauss(120, 3)  # compressor ligado
    elif alimentado:
        potencia = 2.0                   # só a eletrônica do freezer
    else:
        potencia = 0.0                   # sem energia ou relé desligado
    return {
        "minuto": estado["minuto"],
        "temperatura_c": round(estado["temperatura"] + random.gauss(0, 0.05), 2),  # ruído de medição
        "temp_ambiente_c": round(TEMP_AMBIENTE + random.gauss(0, 0.05), 1),        # ruído de medição
        "tensao_v": round(random.gauss(127, 1), 1) if estado["energia"] else 0.0,
        "potencia_w": round(potencia, 1),
        "compressor": "ligado" if potencia > 50 else "desligado",  # inferido pela potência
        "porta": "aberta" if estado["porta_aberta"] else "fechada",
        "rele": "ligado" if estado["rele"] else "desligado",
    }

alarmes_ativos = set()     # alarmes ativos no momento
minutos_porta_aberta = 0   # minutos seguidos com a porta aberta

def verificar_alarmes(leitura):
    """Atualiza os alarmes e retorna os que mudaram de estado nesta leitura."""
    global minutos_porta_aberta

    # Conta os minutos seguidos com a porta aberta (cada leitura = 1 min)
    if leitura["porta"] == "aberta":
        minutos_porta_aberta += 1
        duracao_porta = minutos_porta_aberta
    else:
        duracao_porta = minutos_porta_aberta  # duração da abertura que acabou de terminar
        minutos_porta_aberta = 0

    # Condição de cada alarme nesta leitura
    condicoes = {
        "FALTA_DE_ENERGIA": leitura["tensao_v"] < TENSAO_MINIMA,
        "TEMPERATURA_ALTA": leitura["temperatura_c"] > LIMITE_TEMP,
        "PORTA_ABERTA": minutos_porta_aberta >= PORTA_TEMPO_MAX_MIN,
    }

    # Registra só os alarmes que mudaram de estado
    mudancas = []
    for nome, ativo in condicoes.items():
        if ativo == (nome in alarmes_ativos):
            continue  # sem mudança
        if ativo:
            alarmes_ativos.add(nome)
        else:
            alarmes_ativos.discard(nome)
        alarme = {"alarme": nome,
                  "estado": "ativo" if ativo else "encerrado",
                  "severidade": SEVERIDADE[nome],
                  "minuto": leitura["minuto"],
                  "temperatura_c": leitura["temperatura_c"]}
        if nome == "PORTA_ABERTA":
            alarme["minutos_porta_aberta"] = duracao_porta
        mudancas.append(alarme)
    return mudancas

### Comunicação MQTT

| Tópico | Sentido | QoS | Conteúdo |
|---|---|---|---|
| `.../telemetria` | tomada → nuvem | 0 | leituras a cada 2 s (JSON) |
| `.../alarme` | tomada → nuvem | 1 | alarme ativado ou encerrado (JSON) |
| `.../status` | tomada → nuvem | 1, *retained* | `online` / `offline` |
| `.../cmd` | nuvem → tomada | 1 | comandos em texto puro |

- **QoS:** a telemetria usa QoS 0 porque uma leitura perdida é substituída pela próxima, 2 s depois. Os alarmes usam QoS 1 (entrega garantida) porque não podem se perder.
- **Last Will:** registrado na conexão. Se a tomada cair sem se despedir, o próprio broker publica `offline` no lugar dela.
- **Retained:** o broker guarda o último status, então quem se conecta depois já vê se a tomada está online.

**Comandos.** `ligar` e `desligar` são comandos reais do produto. Os outros só existem na simulação, para provocar os eventos que, no mundo real, aconteceriam fisicamente.

In [ ]:
import json
import paho.mqtt.client as mqtt

# Comando -> (chave do estado, novo valor)
COMANDOS = {
    "ligar":         ("rele", True),           # produto: liga o freezer pelo relé
    "desligar":      ("rele", False),          # produto: desliga o freezer pelo relé
    "abrir_porta":   ("porta_aberta", True),   # simulação
    "fechar_porta":  ("porta_aberta", False),  # simulação
    "falta_energia": ("energia", False),       # simulação
    "volta_energia": ("energia", True),        # simulação
}

LINHA_AVISO = "~" * 115                        # separador dos avisos de conexão e comandos
NEGRITO_AVISO, RESET_AVISO = "\033[1m", "\033[0m"  # códigos ANSI: negrito e fim da formatação

def imprimir_aviso(texto):
    """Imprime um aviso em negrito entre separadores."""
    print(f"{NEGRITO_AVISO}{LINHA_AVISO}")
    print(texto)
    print(f"{LINHA_AVISO}{RESET_AVISO}")

def on_connect(client, userdata, flags, reason_code, properties):
    """Ao conectar: assina o tópico de comandos e publica o status online."""
    imprimir_aviso(f"Conectado ao broker {BROKER} ({reason_code})")
    client.subscribe(f"{TOPICO}/cmd", qos=1)
    client.publish(f"{TOPICO}/status", "online", qos=1, retain=True)

def on_message(client, userdata, msg):
    """Aplica ao estado o comando recebido no tópico cmd."""
    comando = msg.payload.decode().strip().lower()
    if comando in COMANDOS:
        variavel, valor = COMANDOS[comando]
        estado[variavel] = valor
        imprimir_aviso(f"Comando recebido: {comando}")
    else:
        imprimir_aviso(f"Comando desconhecido: {comando!r}. Opções: {', '.join(COMANDOS)}")

def conectar():
    """Cria o cliente MQTT, registra o Last Will e conecta ao broker."""
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.will_set(f"{TOPICO}/status", "offline", qos=1, retain=True)
    client.connect(BROKER, PORTA)
    client.loop_start()  # rede tratada em segundo plano
    return client

### Execução

A cada 2 s, o freezer avança 1 minuto, a tomada lê os sensores e publica a telemetria e os alarmes que mudaram.

**Ao clicar em ■ e parar a execução da célula:** a tomada publica `offline` e desconecta. Executar a célula de novo continua de onde parou; para recomeçar do zero, execute de novo a partir da célula do `estado`.

In [ ]:
import time

LARGURA = 115  # largura das faixas de alarme

# Códigos ANSI de cor e formatação
VERMELHO, LARANJA, VERDE = "\033[91m", "\033[38;5;208m", "\033[92m"
NEGRITO, RESET = "\033[1m", "\033[0m"

# Nomes curtos para a coluna de alarmes
ROTULO_CURTO = {"FALTA_DE_ENERGIA": "energia", "TEMPERATURA_ALTA": "temperatura", "PORTA_ABERTA": "porta"}


def imprimir_alarme(alarme):
    """Imprime o alarme: vermelho (crítico) ou laranja (aviso) se ativo, verde se encerrado."""
    ativo = alarme["estado"] == "ativo"
    detalhes = f"min {alarme['minuto']}, {alarme['temperatura_c']:.2f} °C"
    if "minutos_porta_aberta" in alarme:
        verbo = "aberta há" if ativo else "ficou aberta por"
        detalhes += f", porta {verbo} {alarme['minutos_porta_aberta']} min"
    if ativo:
        if alarme["severidade"] == "critico":
            cor, rotulo = VERMELHO, "[!] ALARME CRÍTICO"
        else:
            cor, rotulo = LARANJA, "[!] AVISO"
        print(f"{cor}{NEGRITO}{'=' * LARGURA}")
        print(f"{rotulo}: {alarme['alarme']} ({detalhes})")
        print(f"{'=' * LARGURA}{RESET}")
    else:
        print(f"{VERDE}{'-' * LARGURA}")
        print(f"[ok] Alarme encerrado: {alarme['alarme']} ({detalhes})")
        print(f"{'-' * LARGURA}{RESET}")


def resumo_alarmes():
    """Retorna o texto da coluna de alarmes ativos, colorido pela gravidade."""
    itens = []
    for nome in SEVERIDADE:  # mesma ordem da configuração
        if nome in alarmes_ativos:
            cor = VERMELHO if SEVERIDADE[nome] == "critico" else LARANJA
            itens.append(f"{cor}{ROTULO_CURTO[nome]}{RESET}")
    return "alarmes: " + (", ".join(itens) if itens else "-")

In [ ]:
client = conectar()
alarmes_ativos.clear()  # reavalia os alarmes do zero, como num reinício do dispositivo

try:
    while True:
        avancar_freezer()
        leitura = ler_sensores()
        mudancas = verificar_alarmes(leitura)  # uma única chamada por leitura

        # Publica e mostra a leitura
        client.publish(f"{TOPICO}/telemetria", json.dumps(leitura), qos=0)
        print(f"min {leitura['minuto']:4d} | {leitura['temperatura_c']:7.2f} °C | "
              f"{leitura['potencia_w']:6.1f} W | {leitura['tensao_v']:6.1f} V | "
              f"porta {leitura['porta']:7s} | relé {leitura['rele']:9s} | {resumo_alarmes()}")

        # Publica e mostra os alarmes que mudaram
        for alarme in mudancas:
            client.publish(f"{TOPICO}/alarme", json.dumps(alarme), qos=1)
            imprimir_alarme(alarme)

        time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    print("Parado pelo usuário.")
finally:
    # Desligamento: publica offline e desconecta
    client.publish(f"{TOPICO}/status", "offline", qos=1, retain=True).wait_for_publish(timeout=2)
    client.disconnect()
    client.loop_stop()

min  185 |  -18.93 °C |    2.0 W |  127.5 V | porta fechada | relé ligado    | alarmes: -
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Conectado ao broker broker.emqx.io (Success)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
min  186 |  -18.79 °C |    2.0 W |  128.6 V | porta fechada | relé ligado    | alarmes: -
min  187 |  -18.63 °C |    2.0 W |  126.3 V | porta fechada | relé ligado    | alarmes: -
min  188 |  -18.46 °C |    2.0 W |  127.7 V | porta fechada | relé ligado    | alarmes: -
min  189 |  -18.26 °C |    2.0 W |  128.9 V | porta fechada | relé ligado    | alarmes: -
min  190 |  -18.15 °C |    2.0 W |  126.8 V | porta fechada | relé ligado    | alarmes: -
min  191 |  -17.91 °C |    2.0 W |  127.7 V | porta fechada | relé ligado    | alarmes: -
min  192 |  -17.89 °C |    2.0 W |  128.2 V | porta fechada | relé ligado    | alarmes: -
min